In [1]:
import math
import numpy as np
import tensorflow as tf

from plotly.subplots import make_subplots
import plotly.graph_objects as go

from optical_communication_system.components import *
from optical_communication_system.datatypes import ElectricalSignal, OpticalSignal
from optical_communication_system.utils import dbm_to_w, from_db
from optical_communication_system.plot import *

2024-11-03 21:20:52.242206: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2024-11-03 21:20:52.281518: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-11-03 21:20:52.281544: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-11-03 21:20:52.282544: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-11-03 21:20:52.288115: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2024-11-03 21:20:52.288925: I tensorflow/core/platform/cpu_feature_guard.cc:1

In [2]:
Component.trace_debug_enabled_by_default = False

data_rate = 10e9
sampling_rate = data_rate * 10
central_frequency = 193.414e12 # Hz

input_power = 2e-3 # Watts

lpf = BandPassFilter(min_frequency=0.0, max_frequency=1 * data_rate, name="lpf")

bits_per_symbol = 2

channel_osnr = 30.0

channel = Composition([
    OpticalAWGN(relative_db=-channel_osnr, name="channel_noise"),
    OpticalFiber(
        attenuation=0.22,
        length=40.0,
        chromatic_dispersion=17.0,
        eff_non_linear_coeff=1.76,
        central_frequency=central_frequency,
        step_count=1000,
        name="fiber"
    )
], name="channel")

input_laser = Laser(power=input_power, name="laser")

photodiode = Composition([Photodiode(), Quantizer(step=1e-4), lpf])

transmitter = Composition([
    PAMEncoder(
        sampling_rate=sampling_rate,
        data_rate=data_rate,
        symbol_encoder=PAMSymbolEncoder(
            bits_per_symbol=bits_per_symbol,
            adjust_for_mzm=True,
            name="pam_symbol_encoder"
        )
    ),
    Composition([lpf], name="encoder_output"),
    MachZehnderModulator(carrier_source=input_laser, name="modulator")
], name="transmitter")

receiver = Composition([
    Composition([photodiode], name="receiver_output"),
    ElectricalGain(gain=1.0),
    PAMDecoder(
        data_rate=data_rate,
        symbol_decoder=PAMSymbolDecoder(
            bits_per_symbol=bits_per_symbol,
            name="pam_symbol_decoder"
        ),
        name="decoder_output"
    )
])

system = Composition([transmitter, channel, receiver])
channel_loss_estimator = ChannelLossEstimator(
    laser=input_laser,
    photodiode=photodiode,
    channel=channel,
    sampling_rate=1e9
)

rng = tf.random.Generator.from_non_deterministic_state(alg="philox")
ber_tester = BERTester(system)

In [3]:
input_signal = tf.constant([[1, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1]], dtype=tf.int32)
#input_signal = rng.uniform((1, 512), 0, 2, dtype=tf.int32)

data_rate = 10e9
sampling_rate = data_rate * 10
central_frequency = 193.414e12 # Hz

input_power = dbm_to_w(3.0) # Watts

bits_per_symbol = 2
channel_osnr = 30.0 * 100
cutoff_frequency = 2.0 * data_rate

system_params = dict(
    laser__power=tf.constant(input_power),
    fiber__step_count=tf.constant(1),
    fiber__length=tf.constant(20.0),
    fiber__eff_non_linear_coeff=tf.constant(1.76 * 0),
    fiber__chromatic_dispersion=tf.constant(17.0 * 2),
    pam_encoder__data_rate=tf.constant(data_rate),
    pam_encoder__sampling_rate=tf.constant(sampling_rate),
    decoder_output__data_rate=tf.constant(data_rate),
    pam_symbol_encoder__bits_per_symbol=tf.constant(bits_per_symbol),
    pam_symbol_decoder__bits_per_symbol=tf.constant(bits_per_symbol),
    channel_noise__relative_db=tf.constant(-channel_osnr),
    lpf__max_frequency=tf.constant(cutoff_frequency),
    quantizer__step=tf.constant(1e-6),
    batch_size=tf.shape(input_signal)[0],
    parameter_dims=tf.constant(1),
    photodiode__sensitivity=dbm_to_w(-20.0),
    photodiode__maximum_power=dbm_to_w(3.0)
)

estimated_channel_loss = channel_loss_estimator(1000e-9, **system_params)[0]

symbol_scale = input_power / estimated_channel_loss
system_params["electrical_gain__gain"] = 1.0 / symbol_scale

%time system.trace(input_signal, **system_params)

2024-11-03 21:20:54.740831: W tensorflow/core/grappler/optimizers/loop_optimizer.cc:933] Skipping loop optimization for Merge node with control input: cond/branch_executed/_8
2024-11-03 21:20:54.821712: W tensorflow/core/grappler/optimizers/loop_optimizer.cc:933] Skipping loop optimization for Merge node with control input: cond/branch_executed/_9


2024-11-03 21:20:55.062971: W tensorflow/core/grappler/optimizers/loop_optimizer.cc:933] Skipping loop optimization for Merge node with control input: cond/branch_executed/_9
2024-11-03 21:20:55.114789: W tensorflow/core/grappler/optimizers/loop_optimizer.cc:933] Skipping loop optimization for Merge node with control input: cond/branch_executed/_9
2024-11-03 21:20:55.755052: W tensorflow/core/grappler/optimizers/loop_optimizer.cc:933] Skipping loop optimization for Merge node with control input: cond/branch_executed/_9
2024-11-03 21:20:56.017799: W tensorflow/core/grappler/optimizers/loop_optimizer.cc:933] Skipping loop optimization for Merge node with control input: cond/branch_executed/_9
2024-11-03 21:20:56.044111: W tensorflow/core/grappler/optimizers/loop_optimizer.cc:933] Skipping loop optimization for Merge node with control input: cond/branch_executed/_9


CPU times: user 804 ms, sys: 12.4 ms, total: 817 ms
Wall time: 813 ms


2024-11-03 21:20:56.596998: W tensorflow/core/grappler/optimizers/loop_optimizer.cc:933] Skipping loop optimization for Merge node with control input: cond/branch_executed/_9


<tf.Tensor: shape=(1, 1, 16), dtype=int32, numpy=array([[[1, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1]]], dtype=int32)>

In [4]:
def plot(chromatic_dispersion):
    params = dict(system_params)
    params["fiber__chromatic_dispersion"] = tf.constant(chromatic_dispersion)
    system.trace(input_signal, **params)

    fig = make_subplots(
        rows=3,
        cols=2,
        subplot_titles=(
            "Input sequence", "Decoded sequence",
            "Encoded signal", "Output signal",
            "Signal after fiber", "Received signal"
        ),
        vertical_spacing = 0.05
    )

    plot_bitseq(input_signal, fig, 0, row=1, col=1)
    plot_bitseq(system.output, fig, row=1, col=2)
    plot_envelope(system.find_child("encoder_output").output, fig, row=2, col=1)
    plot_power(system.find_child("modulator").output, fig, row=2, col=2)
    plot_power(system.find_child("channel").output, fig, row=3, col=1)
    
    plot_envelope(system.find_child("receiver_output").output, fig, row=3, col=2)

    fig.update_layout(
        autosize=True,
        height=1000
    )

    return fig


import ipywidgets

ipywidgets.interact(plot, chromatic_dispersion=(-300.0, 300.0, 1.0))

interactive(children=(FloatSlider(value=0.0, description='chromatic_dispersion', max=300.0, min=-300.0, step=1…

<function __main__.plot(chromatic_dispersion)>

In [5]:
signal = system.find_child("receiver_output").output
fig = plot_eye_diagram(signal, symbol_rate=data_rate / bits_per_symbol, line={"color": "rgba(0, 0, 255, 0.5)"})
fig.show()

In [6]:
%time ber_tester(tf.constant(128), **system_params)

CPU times: user 183 ms, sys: 30.4 ms, total: 213 ms
Wall time: 195 ms


2024-11-03 21:20:57.393226: W tensorflow/core/grappler/optimizers/loop_optimizer.cc:933] Skipping loop optimization for Merge node with control input: cond/branch_executed/_8


<tf.Tensor: shape=(1,), dtype=float32, numpy=array([0.00179688], dtype=float32)>